# Resting-Only Migraine Trait Classifier — Baseline

**Design (per the research pipeline doc):**
- Labels from `Migraine_Control_Demographics.xlsx` (NOT folder prefixes)
- Excludes M2, M6, M18 (medicated); keeps M13 (resting only)
- Reconstructs ~20-s windows by concatenating 10 contiguous 2-s epochs
- Features: relative band power, alpha peak frequency, spectral entropy, wPLI connectivity graph metrics (32-ch 10-20 subset), topographic asymmetry
- Validation: Leave-One-Subject-Out + grouped 5-fold, subject-level AUC
- Models: Logistic Regression (L2), Random Forest, SVM (RBF)

## Cell 1 — Imports

Loads all required libraries:
- **os / re / glob / pathlib**: filesystem navigation and filename parsing
- **numpy / pandas**: array and tabular data handling
- **matplotlib**: plotting (Agg backend for headless rendering)
- **scipy.signal / scipy.stats**: Welch PSD and spectral entropy
- **scikit-learn**: PCA, scaling, models, and evaluation metrics
- **warnings**: suppress non-critical warnings

In [1]:
import os
import re
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy import signal as sp_signal
from scipy.stats import entropy as sp_entropy

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, roc_curve
)

warnings.filterwarnings('ignore')

## Cell 2 — Paths & Constants

Defines the project paths and all preprocessing/feature parameters:
- **RESTING_DIR**: where the preprocessed `.npy` arrays live
- **DEMOGRAPHICS**: the Excel file with subject labels (control/migraine, aura, medication)
- **OUTPUT_DIR**: where results (CSV, plots) are saved
- **EXCLUDED_MEDICATED**: M2, M6, M18 — excluded in the original study due to medication
- **SFREQ = 512 Hz**: native sampling rate of the saved epochs
- **WINDOW_EPOCHS = 10**: 10 × 2-s epochs concatenated → 20-s windows
- **BANDS**: the 5 canonical EEG frequency bands
- **CONN_CHANNELS**: 32-channel 10-20 subset used for wPLI connectivity (all present in the 130-ch montage)

In [2]:
PROJECT_ROOT = Path.cwd()
RESTING_DIR = PROJECT_ROOT / 'data' / 'MIGRAINE_GPU_preprocessed' / 'resting'
DEMOGRAPHICS = PROJECT_ROOT / 'Dataset' / 'Migraine_Control_Demographics.xlsx'
OUTPUT_DIR = PROJECT_ROOT / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

EXCLUDED_MEDICATED = {'M2', 'M6', 'M18'}   #excluded in original study bcz of bad recordings 
SFREQ = 512.0                               #native sampling rate of saved epochs
EPOCH_S = 2.0                               #each saved epoch is 2 s
WINDOW_EPOCHS = 10                          
WINDOW_S = EPOCH_S * WINDOW_EPOCHS

BANDS = {
    'delta': (0.5, 4.0),
    'theta': (4.0, 8.0),
    'alpha': (8.0, 13.0),
    'beta': (13.0, 30.0),
    'gamma': (30.0, 100.0),
}

CONN_CHANNELS = [
    'Fp1', 'Fp2', 'AF7', 'AF8',
    'F7', 'F3', 'Fz', 'F4', 'F8',
    'FC5', 'FC1', 'FC2', 'FC6',
    'T7', 'C3', 'Cz', 'C4', 'T8',
    'CP5', 'CP1', 'CP2', 'CP6',
    'P7', 'P3', 'Pz', 'P4', 'P8',
    'PO9', 'O1', 'Oz', 'O2', 'PO10',
]

RANDOM_STATE = 42

## Cell 3 — Data Loading Functions

Two helper functions:
- **`load_labels()`**: reads the demographics Excel and builds a `{subject_id: label}` map. `C*` → 0 (control), `M*` → 1 (migraine). This is the **correct** label source (not folder prefixes).
- **`discover_resting_files()`**: scans the resting folder for `*_broadband.npy` files and parses each filename into `{file, subject_dir, base_id}`. Handles both `C1_C1_Resting_broadband.npy` and `M10_1_M10_Resting_broadband.npy` naming patterns.

In [3]:
def load_labels():
    """Build subject -> label map from demographics Excel."""
    df = pd.read_excel(DEMOGRAPHICS)
    labels = {}
    for _, row in df.iterrows():
        pid = str(row['P#']).strip()
        if pid.startswith('C'):
            labels[pid] = 0  #control
        else:
            labels[pid] = 1  #migraine
    return labels


def discover_resting_files():
    """Find all resting broadband .npy files and map to subject IDs."""
    files = sorted(glob.glob(str(RESTING_DIR / '*_broadband.npy')))
    records = []
    for f in files:
        name = os.path.basename(f)
        m = re.match(r'^(C\d+|M\d+_\d+)_(.+)_broadband\.npy$', name)
        if not m:
            print(f'  [skip] unrecognized file: {name}')
            continue
        subject_dir = m.group(1)
        base_id = subject_dir.split('_')[0]  # C1, M10, etc.
        records.append({
            'file': f,
            'subject_dir': subject_dir,
            'base_id': base_id,
        })
    return records

## Cell 4 — Spectral Feature Helpers (PSD, Band Power, Alpha Peak, Entropy)

Four functions that compute per-channel spectral features from a window:
- **`welch_psd()`**: Welch's PSD estimate (4-s FFT windows, 2-s overlap) → `(freqs, psd)`
- **`band_power_features()`**: relative power in each of the 5 bands, normalized by total power → `(n_ch, 5)`
- **`alpha_peak_frequency()`**: the frequency (8–13 Hz) with maximum power per channel → `(n_ch,)`
- **`spectral_entropy_features()`**: normalized spectral entropy per channel → `(n_ch,)`

In [ ]:
def welch_psd(data, sfreq=SFREQ):
    """Compute Welch PSD for each channel. data: (n_ch, n_samples)."""
    freqs, psd = sp_signal.welch(
        data, fs=sfreq, nperseg=int(4 * sfreq),
        noverlap=int(2 * sfreq), axis=-1
    )
    return freqs, psd


def band_power_features(psd, freqs):
    """Relative band power per channel. Returns (n_ch, n_bands)."""
    total = np.trapezoid(psd, freqs, axis=-1)  # (n_ch,)
    total[total == 0] = 1e-12
    feats = []
    for band, (lo, hi) in BANDS.items():
        mask = (freqs >= lo) & (freqs <= hi)
        bp = np.trapezoid(psd[:, mask], freqs[mask], axis=-1)
        feats.append(bp / total)
    return np.stack(feats, axis=-1)  # (n_ch, n_bands)


def alpha_peak_frequency(psd, freqs):
    """Alpha peak frequency per channel (8-13 Hz)."""
    mask = (freqs >= 8.0) & (freqs <= 13.0)
    f_alpha = freqs[mask]
    p_alpha = psd[:, mask]
    peaks = f_alpha[np.argmax(p_alpha, axis=-1)]
    return peaks  


def spectral_entropy_features(psd):
    """Normalized spectral entropy per channel."""
    p = psd / (psd.sum(axis=-1, keepdims=True) + 1e-12)
    return sp_entropy(p, axis=-1) / np.log(psd.shape[-1])  

## Cell 5 — Connectivity Helpers (wPLI + Graph Metrics)

These target the **coherence abnormalities** the dataset's own paper (Chamanzar et al., 2020) identified:
- **`wpli_matrix()`**: Weighted Phase Lag Index between all channel pairs. Uses the Hilbert transform to get instantaneous phase, then computes the wPLI — a volume-conduction-robust measure of phase synchronization (Vinck et al., 2011). Returns a symmetric `(n_ch, n_ch)` matrix.
- **`connectivity_graph_features()`**: summarizes the wPLI matrix into graph metrics: mean/std/max/min/median of all pair weights, plus the mean weighted clustering coefficient.

In [5]:
def wpli_matrix(data, sfreq=SFREQ):
    """Weighted Phase Lag Index between all channel pairs.
    data: (n_ch, n_samples). Returns (n_ch, n_ch) wPLI matrix."""
    n_ch = data.shape[0]
    # Hilbert transform to get analytic phase
    analytic = sp_signal.hilbert(data, axis=-1)
    phase = np.angle(analytic)
    wpli = np.zeros((n_ch, n_ch))
    for i in range(n_ch):
        for j in range(i + 1, n_ch):
            dphi = phase[i] - phase[j]
            imag = np.sin(dphi)
            num = np.abs(np.mean(imag))
            den = np.mean(np.abs(imag))
            wpli[i, j] = wpli[j, i] = num / (den + 1e-12)
    return wpli


def connectivity_graph_features(wpli):
    """Graph metrics from wPLI matrix: mean, std, clustering, path length."""
    n = wpli.shape[0]
    triu = wpli[np.triu_indices(n, k=1)]
    feats = {
        'conn_mean': triu.mean(),
        'conn_std': triu.std(),
        'conn_max': triu.max(),
        'conn_min': triu.min(),
        'conn_median': np.median(triu),
    }
    # Clustering coefficient (weighted)
    w = wpli.copy()
    np.fill_diagonal(w, 0)
    deg = w.sum(axis=1)
    denom = deg * (deg - 1)
    denom[denom == 0] = 1e-12
    clustering = np.zeros(n)
    for i in range(n):
        neighbors = np.where(w[i] > 0)[0]
        if len(neighbors) < 2:
            continue
        sub = w[np.ix_(neighbors, neighbors)]
        clustering[i] = sub.sum() / (len(neighbors) * (len(neighbors) - 1) + 1e-12)
    feats['conn_clustering'] = clustering.mean()
    return feats

## Cell 6 — Topographic Asymmetry + Master Feature Extractor

- **`topographic_asymmetry()`**: computes left–right and anterior–posterior power ratios for each band — captures hemispheric and fronto-occipital imbalances.
- **`extract_window_features()`**: the master function that combines **all** features for one 20-s window into a flat dict:
  - 20 band-power stats (mean/std/max/min × 5 bands, log-transformed)
  - 4 alpha-peak-frequency stats
  - 2 spectral-entropy stats
  - 6 connectivity graph metrics (if ≥16 of the 32 channels are present)
  - 10 topographic asymmetry ratios (LR/AP × 5 bands)
  → **42 features total**

In [6]:
def topographic_asymmetry(band_power, ch_names):
    """Left-right and anterior-posterior power ratios."""
    idx = {ch: i for i, ch in enumerate(ch_names)}
    left = ['Fp1', 'F7', 'F3', 'T7', 'C3', 'P7', 'P3', 'O1']
    right = ['Fp2', 'F8', 'F4', 'T8', 'C4', 'P8', 'P4', 'O2']
    ant = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8']
    post = ['P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']
    feats = {}
    for b, band_name in enumerate(BANDS.keys()):
        lr = np.mean([band_power[idx[c], b] for c in left if c in idx]) / \
             (np.mean([band_power[idx[c], b] for c in right if c in idx]) + 1e-12)
        ap = np.mean([band_power[idx[c], b] for c in ant if c in idx]) / \
             (np.mean([band_power[idx[c], b] for c in post if c in idx]) + 1e-12)
        feats[f'LR_{band_name}'] = lr
        feats[f'AP_{band_name}'] = ap
    return feats


def extract_window_features(window_data, ch_names):
    """Extract all features from one 20-s window.
    window_data: (n_ch, n_samples). Returns dict of scalar features."""
    feats = {}

    # PSD-based features
    freqs, psd = welch_psd(window_data)
    bp = band_power_features(psd, freqs)          # (n_ch, 5)
    apf = alpha_peak_frequency(psd, freqs)        # (n_ch,)
    se = spectral_entropy_features(psd)           # (n_ch,)

    # Flatten band power (log-transformed)
    for b, band_name in enumerate(BANDS.keys()):
        feats[f'bp_{band_name}_mean'] = np.log1p(bp[:, b].mean())
        feats[f'bp_{band_name}_std'] = np.log1p(bp[:, b].std())
        feats[f'bp_{band_name}_max'] = np.log1p(bp[:, b].max())
        feats[f'bp_{band_name}_min'] = np.log1p(bp[:, b].min())

    # Alpha peak frequency stats
    feats['apf_mean'] = apf.mean()
    feats['apf_std'] = apf.std()
    feats['apf_max'] = apf.max()
    feats['apf_min'] = apf.min()

    # Spectral entropy stats
    feats['se_mean'] = se.mean()
    feats['se_std'] = se.std()

    # Connectivity on 32-ch subset
    conn_idx = [ch_names.index(c) for c in CONN_CHANNELS if c in ch_names]
    if len(conn_idx) >= 16:
        wpli = wpli_matrix(window_data[conn_idx])
        feats.update(connectivity_graph_features(wpli))

    # Topographic asymmetry
    feats.update(topographic_asymmetry(bp, ch_names))

    return feats

## Cell 7 — Load Labels & Discover Files

Runs the two data-loading functions and applies the **exclusion criteria**:
- **M2, M6, M18** excluded (medicated — per the original study)
- **C2, C6, C12** excluded (not in the demographics Excel — they were extra controls not used in the matched analysis)
- **M13 kept** (only SSAEP is missing; resting is fine)

Expected result: **33 recordings** (18 control + 15 migraine) after exclusions.

In [7]:
# 1. Labels
labels = load_labels()
print(f'[1] Loaded {len(labels)} subject labels from demographics.')

# 2. Discover files
records = discover_resting_files()
print(f'[2] Found {len(records)} resting broadband files.')

# 3. Filter by exclusion criteria
kept = []
for r in records:
    base = r['base_id']
    if base in EXCLUDED_MEDICATED:
        print(f'    [excluded] {r["subject_dir"]} (medicated)')
        continue
    if base not in labels:
        print(f'    [excluded] {r["subject_dir"]} (no demographics)')
        continue
    kept.append(r)
print(f'    Kept {len(kept)} recordings after exclusions.')

[1] Loaded 36 subject labels from demographics.
[2] Found 39 resting broadband files.
    [excluded] C12 (no demographics)
    [excluded] C2 (no demographics)
    [excluded] C6 (no demographics)
    [excluded] M18_1 (medicated)
    [excluded] M2_1 (medicated)
    [excluded] M6_1 (medicated)
    Kept 33 recordings after exclusions.


## Cell 8 — Build 20-s Windows & Extract Features

For each kept recording:
1. Loads the `(n_epochs, 130, 1024)` broadband array
2. Concatenates every **10 contiguous 2-s epochs** → one 20-s window `(130, 10240)`
3. Extracts the 42 features per window via `extract_window_features()`
4. Records the subject label and subject group (for leakage-free CV)

Also saves `resting_subject_stats.csv` with per-subject epoch/window counts.

Expected: **~608 windows** (330 control + 278 migraine) across **33 subjects**.

In [8]:
print('\n[3] Extracting features from 20-s windows...')
X_list, y_list, groups_list = [], [], []
subject_stats = []
feat_names = None

for r in kept:
    arr = np.load(r['file'])  # (n_epochs, 130, 1024)
    n_epochs = arr.shape[0]
    n_windows = n_epochs // WINDOW_EPOCHS
    if n_windows == 0:
        print(f'    [skip] {r["subject_dir"]}: only {n_epochs} epochs (< {WINDOW_EPOCHS})')
        continue

    # Load channel names for this recording
    ch_file = RESTING_DIR / f'channel_names_{os.path.basename(r["file"]).replace("_broadband.npy", "")}.txt'
    if not ch_file.exists():
        # Fallback: use the top-level channel_names.txt
        ch_file = RESTING_DIR.parent / 'channel_names.txt'
    with open(ch_file) as fh:
        ch_names = [l.strip() for l in fh if l.strip()]

    label = labels[r['base_id']]
    n_feat = None
    for w in range(n_windows):
        start = w * WINDOW_EPOCHS
        end = start + WINDOW_EPOCHS
        window_data = arr[start:end].transpose(1, 0, 2).reshape(arr.shape[1], -1)  # (130, 20s*512)
        feats = extract_window_features(window_data, ch_names)
        if n_feat is None:
            n_feat = len(feats)
            feat_names = list(feats.keys())
        X_list.append(list(feats.values()))
        y_list.append(label)
        groups_list.append(r['subject_dir'])

    subject_stats.append({
        'subject': r['subject_dir'],
        'group': 'migraine' if label == 1 else 'control',
        'n_epochs': n_epochs,
        'n_windows': n_windows,
    })
    print(f'    {r["subject_dir"]}: {n_epochs} epochs -> {n_windows} windows (label={label})')

X = np.array(X_list, dtype=np.float64)
y = np.array(y_list)
groups = np.array(groups_list)
print(f'\n    Total windows: {X.shape[0]}, features: {X.shape[1]}')
print(f'    Migraine windows: {(y == 1).sum()}, Control windows: {(y == 0).sum()}')
print(f'    Subjects: {len(np.unique(groups))}')

# Save subject stats
pd.DataFrame(subject_stats).to_csv(OUTPUT_DIR / 'resting_subject_stats.csv', index=False)


[3] Extracting features from 20-s windows...
    C10: 200 epochs -> 20 windows (label=0)
    C11: 182 epochs -> 18 windows (label=0)
    C13: 181 epochs -> 18 windows (label=0)
    C14: 178 epochs -> 17 windows (label=0)
    C15: 188 epochs -> 18 windows (label=0)
    C16: 154 epochs -> 15 windows (label=0)
    C17: 157 epochs -> 15 windows (label=0)
    C18: 187 epochs -> 18 windows (label=0)
    C19: 202 epochs -> 20 windows (label=0)
    C1: 193 epochs -> 19 windows (label=0)
    C20: 174 epochs -> 17 windows (label=0)
    C21: 189 epochs -> 18 windows (label=0)
    C3: 159 epochs -> 15 windows (label=0)
    C4: 204 epochs -> 20 windows (label=0)
    C5: 193 epochs -> 19 windows (label=0)
    C7: 269 epochs -> 26 windows (label=0)
    C8: 194 epochs -> 19 windows (label=0)
    C9: 187 epochs -> 18 windows (label=0)
    M10_1: 163 epochs -> 16 windows (label=1)
    M11_1: 178 epochs -> 17 windows (label=1)
    M12_2: 184 epochs -> 18 windows (label=1)
    M13_2: 168 epochs -> 16 win

## Cell 9 — Standardize + PCA

Reduces the 42 features to a lower-dimensional space:
- **StandardScaler**: z-score normalization (mean 0, std 1) so features are comparable
- **PCA**: keeps up to 50 components (here all 42, since 42 < 50) — captures the full variance

Note: the scaler here is fit on **all** data for the PCA step. The **per-fold scaler** inside the LOSO loop (next cell) is the leakage-safe one.

In [9]:
print('\n[4] Standardizing and reducing features (PCA)...')
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
pca = PCA(n_components=min(50, X.shape[1]), random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
print(f'    PCA explained variance: {pca.explained_variance_ratio_.sum():.3f} '
      f'({pca.n_components_} components)')


[4] Standardizing and reducing features (PCA)...
    PCA explained variance: 1.000 (42 components)


## Cell 10 — Define Models

Three classical ML classifiers (chosen for robustness at small n):
- **Logistic Regression (L2)**: linear baseline, interpretable, low variance
- **Random Forest**: non-linear, handles feature interactions, gives feature importance
- **SVM (RBF)**: non-linear kernel, good for small-sample high-dim data

In [10]:
models = {
    'LogisticRegression': LogisticRegression(
        C=1.0, max_iter=2000, random_state=RANDOM_STATE),
    'RandomForest': RandomForestClassifier(
        n_estimators=300, max_depth=8, random_state=RANDOM_STATE),
    'SVM_RBF': SVC(
        C=1.0, kernel='rbf', gamma='scale', probability=True,
        random_state=RANDOM_STATE),
}

## Cell 11 — Leave-One-Subject-Out (LOSO) Evaluation

The **most leakage-safe** validation strategy for small datasets:
- For each of the 33 subjects, train on the other 32 and test on that one subject
- **All windows from one subject stay in one fold** — no subject leakage
- A **fresh StandardScaler is fit on train folds only** (leakage prevention)
- Window-level predictions are **aggregated to subject level** (mean probability per subject) for the final metrics

Metrics reported: subject-level AUC, accuracy, precision, recall, F1, and confusion matrix.

In [11]:
print('\n[5] Leave-One-Subject-Out evaluation...')
logo = LeaveOneGroupOut()
results = []
roc_data = {}

for model_name, model in models.items():
    print(f'\n  --- {model_name} ---')
    y_true_all, y_prob_all, y_pred_all = [], [], []
    fold_aucs = []
    fold_accs = []
    fold_f1s = []

    for train_idx, test_idx in logo.split(X_pca, y, groups):
        X_tr, X_te = X_pca[train_idx], X_pca[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]

        # Fit scaler on train only (leakage prevention)
        scaler_fold = StandardScaler()
        X_tr_s = scaler_fold.fit_transform(X_tr)
        X_te_s = scaler_fold.transform(X_te)

        model.fit(X_tr_s, y_tr)
        y_prob = model.predict_proba(X_te_s)[:, 1]
        y_pred = model.predict(X_te_s)

        y_true_all.extend(y_te)
        y_prob_all.extend(y_prob)
        y_pred_all.extend(y_pred)

        if len(np.unique(y_te)) > 1:
            fold_aucs.append(roc_auc_score(y_te, y_prob))
        fold_accs.append(accuracy_score(y_te, y_pred))
        fold_f1s.append(f1_score(y_te, y_pred, zero_division=0))

    # Subject-level aggregation (mean probability per subject)
    subj_prob = {}
    subj_true = {}
    for g, p, t in zip(groups, y_prob_all, y_true_all):
        subj_prob.setdefault(g, []).append(p)
        subj_true[g] = t
    subj_prob_mean = np.array([np.mean(v) for v in subj_prob.values()])
    subj_true_arr = np.array([subj_true[g] for g in subj_prob.keys()])
    subj_pred = (subj_prob_mean >= 0.5).astype(int)

    subj_auc = roc_auc_score(subj_true_arr, subj_prob_mean)
    subj_acc = accuracy_score(subj_true_arr, subj_pred)
    subj_prec = precision_score(subj_true_arr, subj_pred, zero_division=0)
    subj_rec = recall_score(subj_true_arr, subj_pred, zero_division=0)
    subj_f1 = f1_score(subj_true_arr, subj_pred, zero_division=0)
    cm = confusion_matrix(subj_true_arr, subj_pred)

    results.append({
        'model': model_name,
        'window_auc': np.mean(fold_aucs) if fold_aucs else np.nan,
        'window_auc_std': np.std(fold_aucs) if fold_aucs else np.nan,
        'subject_auc': subj_auc,
        'subject_accuracy': subj_acc,
        'subject_precision': subj_prec,
        'subject_recall': subj_rec,
        'subject_f1': subj_f1,
        'n_subjects': len(subj_true_arr),
        'n_migraine': int(subj_true_arr.sum()),
        'n_control': int((1 - subj_true_arr).sum()),
        'cm_tn': int(cm[0, 0]), 'cm_fp': int(cm[0, 1]),
        'cm_fn': int(cm[1, 0]), 'cm_tp': int(cm[1, 1]),
    })

    # ROC curve for plotting
    fpr, tpr, _ = roc_curve(subj_true_arr, subj_prob_mean)
    roc_data[model_name] = (fpr, tpr, subj_auc)

    print(f'    Window-level AUC: {np.mean(fold_aucs):.3f} ± {np.std(fold_aucs):.3f}')
    print(f'    Subject-level AUC: {subj_auc:.3f}')
    print(f'    Subject-level Acc: {subj_acc:.3f} | Prec: {subj_prec:.3f} | '
          f'Rec: {subj_rec:.3f} | F1: {subj_f1:.3f}')
    print(f'    Confusion matrix (TN FP / FN TP): {cm.tolist()}')

results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_DIR / 'resting_baseline_results.csv', index=False)
print(f'\n[6] Results saved to {OUTPUT_DIR / "resting_baseline_results.csv"}')


[5] Leave-One-Subject-Out evaluation...

  --- LogisticRegression ---
    Window-level AUC: nan ± nan
    Subject-level AUC: 0.559
    Subject-level Acc: 0.545 | Prec: 0.500 | Rec: 0.467 | F1: 0.483
    Confusion matrix (TN FP / FN TP): [[11, 7], [8, 7]]

  --- RandomForest ---
    Window-level AUC: nan ± nan
    Subject-level AUC: 0.300
    Subject-level Acc: 0.333 | Prec: 0.000 | Rec: 0.000 | F1: 0.000
    Confusion matrix (TN FP / FN TP): [[11, 7], [15, 0]]

  --- SVM_RBF ---
    Window-level AUC: nan ± nan
    Subject-level AUC: 0.515
    Subject-level Acc: 0.545 | Prec: 0.500 | Rec: 0.400 | F1: 0.444
    Confusion matrix (TN FP / FN TP): [[12, 6], [9, 6]]

[6] Results saved to g:\Study\FYDP-I_Personalized-Migraine-Mitigation-Via-Binaural-Beats\output\resting_baseline_results.csv


## Cell 12 — Plot Subject-Level ROC Curves

Plots the ROC curve for each model using **subject-level** predictions (mean window probability per subject), with the chance diagonal for reference. Saves to `output/resting_baseline_roc.png`.

In [12]:
plt.figure(figsize=(8, 6))
for name, (fpr, tpr, auc) in roc_data.items():
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC={auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Chance')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Resting-Only Migraine Trait Detection — Subject-Level ROC')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'resting_baseline_roc.png', dpi=150)
plt.show()
print(f'    ROC plot saved to {OUTPUT_DIR / "resting_baseline_roc.png"}')

    ROC plot saved to g:\Study\FYDP-I_Personalized-Migraine-Mitigation-Via-Binaural-Beats\output\resting_baseline_roc.png


## Cell 13 — Feature Importance (Random Forest)

Fits a Random Forest on the **full 42-feature** set (not PCA-reduced) and reports the top-20 features by Gini importance. This tells you **which spectral/connectivity/asymmetry features actually drive the migraine-vs-control decision** — a scientifically interpretable output. Saves to `output/resting_feature_importance.csv`.

In [13]:
print('\n[7] Feature importance (Random Forest, top 20)...')
rf_imp = RandomForestClassifier(n_estimators=500, max_depth=8, random_state=RANDOM_STATE)
rf_imp.fit(X_scaled, y)
importances = pd.Series(rf_imp.feature_importances_, index=feat_names)
importances = importances.sort_values(ascending=False).head(20)
print(importances.to_string())
importances.to_csv(OUTPUT_DIR / 'resting_feature_importance.csv')


[7] Feature importance (Random Forest, top 20)...
bp_beta_mean     0.055174
bp_delta_std     0.050067
LR_gamma         0.047630
LR_beta          0.046205
LR_delta         0.045766
bp_theta_min     0.041293
se_mean          0.039213
bp_gamma_std     0.037053
apf_mean         0.033150
AP_delta         0.031674
bp_beta_max      0.030307
bp_delta_mean    0.028169
bp_beta_std      0.027817
bp_alpha_std     0.027703
AP_gamma         0.026110
bp_beta_min      0.025987
bp_theta_mean    0.025475
bp_alpha_min     0.024670
AP_beta          0.023813
bp_alpha_mean    0.023629


## Cell 14 — Summary & Chance-Level Reference

Prints the final subject-level metrics table and the **chance-level accuracy bound** (Combrisson & Jerbi, 2015):
- With n=33 subjects (15 migraine / 18 control), chance accuracy = 0.500 with 95% CI ≈ [0.329, 0.671]
- Any subject-level accuracy within this CI is **not** statistically above chance
- This is the honest statistical framing required for small-n EEG classification

In [14]:
print('\n' + '=' * 70)
print('SUMMARY')
print('=' * 70)
print(results_df[['model', 'subject_auc', 'subject_accuracy', 'subject_precision',
                  'subject_recall', 'subject_f1']].to_string(index=False))
print('\nChance-level reference (Combrisson & Jerbi 2015):')
n_subj = results_df['n_subjects'].iloc[0]
n_pos = results_df['n_migraine'].iloc[0]
chance_acc = 0.5
se = np.sqrt(chance_acc * (1 - chance_acc) / n_subj)
print(f'  n={n_subj} subjects ({n_pos} migraine / {n_subj - n_pos} control)')
print(f'  Chance accuracy = 0.500, 95% CI = [{chance_acc - 1.96*se:.3f}, {chance_acc + 1.96*se:.3f}]')
print('\nDone.')


SUMMARY
             model  subject_auc  subject_accuracy  subject_precision  subject_recall  subject_f1
LogisticRegression     0.559259          0.545455                0.5        0.466667    0.482759
      RandomForest     0.300000          0.333333                0.0        0.000000    0.000000
           SVM_RBF     0.514815          0.545455                0.5        0.400000    0.444444

Chance-level reference (Combrisson & Jerbi 2015):
  n=33 subjects (15 migraine / 18 control)
  Chance accuracy = 0.500, 95% CI = [0.329, 0.671]

Done.
